In [1]:
import pandas as pd
import glob

files = glob.glob("Data/extracted_data/group*/experiment*/subject*.csv")
df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

print(df.shape)
df.head()
# df.to_parquet("Data/initial_data.parquet", engine="pyarrow", index=False)

(245703, 30)


### Add Sensor Data

The following cells were executed in Snellius cluster where all the sensors_data folders for each subject are stored. As an output we receive the same df + columns of sensors data. It will output the file "initial_sensors_data.parquet".

In [7]:
df = pd.read_parquet("Data/initial_data.parquet", engine="pyarrow")

In [8]:
import glob
import json
import pandas as pd
import numpy as np
from datetime import datetime
from loguru import logger

In [9]:
def build_sensor_intervals(sensor_folder):
    sensor_files = glob.glob(sensor_folder)

    intervals = []
    data_cache = {}

    for f in sensor_files:
        with open(f, "r") as fp:
            data = json.load(fp)

        times = []

        for sensor in data["data"].values():
            for entry in sensor:                
                times.append(datetime.strptime(entry["timestamp"], "%H:%M:%S:%f"))

        if not times:
            continue

        start = min(times)
        end = max(times)

        intervals.append((f, start, end))
        data_cache[f] = data

    # sort by start time
    intervals.sort(key=lambda x: x[1])

    return intervals, data_cache

In [10]:
# def find_sensor_interval(target_time, intervals, idx):
#     """
#     idx = pointer (so we don't scan from start every time)
#     """
#     n = len(intervals)

#     # move forward while target is beyond current interval
#     while idx < n - 1 and target_time > intervals[idx][2]:
#         idx += 1

#     f, start, end = intervals[idx]

#     if start <= target_time <= end:
#         return f, idx

#     return None, idx

def find_sensor_interval(target_time, intervals, idx, max_diff_sec=0.5):
    n = len(intervals)

    # Move pointer forward
    while idx < n - 1 and target_time > intervals[idx][2]:
        idx += 1

    candidates = []

    # current
    f, start, end = intervals[idx]
    dist_current = min(
        abs((target_time - start).total_seconds()),
        abs((target_time - end).total_seconds())
    )
    candidates.append((dist_current, idx))

    # previous
    if idx > 0:
        f_prev, s_prev, e_prev = intervals[idx - 1]
        dist_prev = min(
            abs((target_time - s_prev).total_seconds()),
            abs((target_time - e_prev).total_seconds())
        )
        candidates.append((dist_prev, idx - 1))

    # next
    if idx < n - 1:
        f_next, s_next, e_next = intervals[idx + 1]
        dist_next = min(
            abs((target_time - s_next).total_seconds()),
            abs((target_time - e_next).total_seconds())
        )
        candidates.append((dist_next, idx + 1))

    # pick closest
    best_dist, best_idx = min(candidates, key=lambda x: x[0])

    # if the frame does not correspond to any of the intervals, assign it to the closest one 
    # as long as the difference is not higher than threshold (1 second)
    if best_dist > max_diff_sec:
        return None, idx

    f_best = intervals[best_idx][0]

    return f_best, best_idx

In [11]:
# overall strength of movement, independent of direction
def compute_magnitude(x, y, z):
    return np.sqrt(x**2 + y**2 + z**2)

def aggregate_sensor(sensor_data, sensor_name):
    features = {}

    if len(sensor_data) == 0:
        return features

    values = {}
    for key in sensor_data[0].keys():
        if key.startswith("value"):
            values[key] = np.array([d[key] for d in sensor_data])

    if sensor_name in ["samsung_linear_acceleration_sensor", "lsm6dso_gyroscope"]:
        x, y, z = values["value0"], values["value1"], values["value2"]
        mag = compute_magnitude(x, y, z)

        features.update({
            f"{sensor_name}_mean_x": x.mean(),
            f"{sensor_name}_std_x": x.std(),
            f"{sensor_name}_mean_y": y.mean(),
            f"{sensor_name}_std_y": y.std(),
            f"{sensor_name}_mean_z": z.mean(),
            f"{sensor_name}_std_z": z.std(),
            f"{sensor_name}_mean_mag": mag.mean(),
            f"{sensor_name}_std_mag": mag.std(),
        })

    elif sensor_name == "samsung_rotation_vector":
        for k, v in values.items():
            features[f"{sensor_name}_mean_{k}"] = v.mean()
            features[f"{sensor_name}_std_{k}"] = v.std()

    elif sensor_name == "opt3007_light":
        v = values["value0"]
        features[f"{sensor_name}_mean"] = v.mean()
        features[f"{sensor_name}_std"] = v.std()
        
    elif sensor_name == "samsung_hr_none_wakeup_sensor":
        v = values["value0"]
        features[f"{sensor_name}_value"] = v.mean()
    return features

def aggregate_all_sensors(data):
    all_features = {}
    for sensor_name, sensor_data in data["data"].items():
        feats = aggregate_sensor(sensor_data, sensor_name)
        all_features.update(feats)

    return all_features

In [12]:
def add_sensors_to_df(df, sensor_folder):
    intervals, data_cache = build_sensor_intervals(sensor_folder)

    sensor_rows = []
    idx = 0  # pointer

    for i, row in df.iterrows():
        target_time = datetime.strptime(row["time"], "%H:%M:%S.%f")

        sensor_file, idx = find_sensor_interval(target_time, intervals, idx)

        if sensor_file is None:
            sensor_rows.append({})
            logger.warning(f"No close sensor match for time {target_time}")
            continue

        sensor_json = data_cache[sensor_file]
        features = aggregate_all_sensors(sensor_json)
        sensor_rows.append(features)

    sensor_df = pd.DataFrame(sensor_rows)

    return pd.concat([df.reset_index(drop=True), sensor_df], axis=1)

In [13]:
# example (stored locally)
sub10ex2 = df[(df['subject'] == 'subject_10') & (df['experiment'] == 'experiment02') & (df['group'] == 'group01')]
sensor_folder = r"Data\DIPSER\group01\experiment02\subject_10\watch_sensors\*.json"
sub10ex2_enriched = add_sensors_to_df(sub10ex2, sensor_folder)
sub10ex2_enriched

,group,time,subject,experiment,image_path,metadata,labeler_02 emotion,labeler_02 attention,labeler_01 attention,labeler_04 attention,...,opt3007_light_std,samsung_linear_acceleration_sensor_mean_x,samsung_linear_acceleration_sensor_std_x,samsung_linear_acceleration_sensor_mean_y,samsung_linear_acceleration_sensor_std_y,samsung_linear_acceleration_sensor_mean_z,samsung_linear_acceleration_sensor_std_z,samsung_linear_acceleration_sensor_mean_mag,samsung_linear_acceleration_sensor_std_mag,samsung_hr_none_wakeup_sensor_value
0,group01,10:59:49.008949,subject_10,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,7.0,4.0,5.0,3.0,...,168.366980,-0.043910,0.239511,0.286730,0.902338,0.216460,0.666781,0.913279,0.783004,73.0
1,group01,10:59:49.117509,subject_10,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,168.366980,-0.043910,0.239511,0.286730,0.902338,0.216460,0.666781,0.913279,0.783004,73.0
2,group01,10:59:49.251676,subject_10,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,168.366980,-0.043910,0.239511,0.286730,0.902338,0.216460,0.666781,0.913279,0.783004,73.0
3,group01,10:59:49.344585,subject_10,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,168.366980,-0.043910,0.239511,0.286730,0.902338,0.216460,0.666781,0.913279,0.783004,73.0
4,group01,10:59:49.436001,subject_10,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,139.757790,0.034333,0.209645,0.106303,0.616854,-0.195463,0.349149,0.636550,0.438011,74.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2191,group01,11:04:48.553386,subject_10,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,13.287588,0.042952,0.072887,0.014676,0.177890,-0.036823,0.246840,0.285008,0.141684,73.0
2192,group01,11:04:48.686729,subject_10,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,13.287588,0.042952,0.072887,0.014676,0.177890,-0.036823,0.246840,0.285008,0.141684,73.0
2193,group01,11:04:48.777590,subject_10,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,13.287588,0.042952,0.072887,0.014676,0.177890,-0.036823,0.246840,0.285008,0.141684,73.0
2194,group01,11:04:48.872595,subject_10,experiment02,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,/gpfs/work5/0/prjs2039/DIPSER/group01/experime...,NaN,NaN,NaN,NaN,...,13.287588,0.042952,0.072887,0.014676,0.177890,-0.036823,0.246840,0.285008,0.141684,73.0


## Extract metadata (gender, age, race)

The following cell was executed in Snellius cluster where all the metadata folders for each subject are stored. As an output we receive the same df + 3 columns of age, gender and race for each row. It will output the file "initial_sensors_metadata.parquet" which we use in the subsequent cells.

In [14]:
# Metadata extraction function
def extract_metadata(json_path):
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        face = data.get("person", {}).get("face", {})

        age = face.get("age", None)
        gender = face.get("gender", {}).get("gender_name", None)
        race = face.get("race", {}).get("dominant_race", None)

        # race probabilities
        prob_race = face.get("race", {}).get("probability_race", {})

        return (
            age,
            gender,
            race,
            prob_race.get("asian", None),
            prob_race.get("indian", None),
            prob_race.get("black", None),
            prob_race.get("white", None),
            prob_race.get("middle eastern", None),
            prob_race.get("latino hispanic", None),
        )

    except Exception:
        return (None,) * 9

# Parallel processing
def process_dataframe(df, n_jobs):
    paths = df["metadata"].tolist()
    results = []

    for i, p in enumerate(paths):
        results.append(extract_metadata(p))

        if i % 5000 == 0:
            print(f"Processed {i}/{len(paths)} rows")

    df[[
    "age",
    "gender_name",
    "race",
    "race_asian",
    "race_indian",
    "race_black",
    "race_white",
    "race_middle_eastern",
    "race_latino_hispanic"
    ]] = pd.DataFrame(results, index=df.index)
    return df

def main():
    start_time = time.time()
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True, help="Input dataframe path (parquet)")
    parser.add_argument("--output", required=True, help="Output dataframe path")
    parser.add_argument("--n_jobs", type=int, default=16)

    args = parser.parse_args()

    print("Loading dataframe...")
    df = pd.read_parquet(args.input)

    print(f"Processing {len(df)} rows with {args.n_jobs} workers...")

    df = process_dataframe(df, args.n_jobs)

    print("Saving output...")
    df.to_parquet(args.output)

    print("Done.")
    print(f"Elapsed: {time.time() - start_time:.2f}s")

# df.to_parquet("Data/initial_sensors_metadata.parquet", engine="pyarrow")

### Transform Metadata

In [5]:
df = pd.read_parquet("Data/initial_sensors_metadata.parquet", engine="pyarrow")

In [10]:
df.shape

(487432, 66)

In [6]:
race_cols = [
    "race_asian",
    "race_indian",
    "race_black",
    "race_white",
    "race_middle_eastern",
    "race_latino_hispanic"
]

In [7]:
subject_probs = (
    df.groupby(["group", "subject"])[race_cols]
    .mean()
    .reset_index()
)

In [18]:
"""Deepface extracts per each timeframe the estimation for race, gender and age.
 For the column of gender, we will keep the most common label for the whole subject."""

# we will groupby group experiment and subject and we will keep the most dominant 
def get_mode(series):
    return series.dropna().mode().iloc[0] if not series.dropna().empty else None

In [19]:
def get_dominant_race(row):
    return row[race_cols].idxmax().replace("race_", "")

In [20]:
subject_probs["race"] = subject_probs.apply(get_dominant_race, axis=1)
subject_probs = subject_probs.drop(columns=race_cols)

In [21]:
subject_metadata = (
    df.groupby(["group", "subject"])
    .agg({
        "gender_name": get_mode,
        "age": "mean"
    })
    .reset_index())

# merge with race
subject_metadata = subject_metadata.merge(
    subject_probs,
    on=["group", "subject"],
    how="left"
)

In [22]:
df = df.drop(columns=['race', 'gender_name', 'age']) # will be replaced with the new values
df = df.merge(
    subject_metadata,
    on=["group", "subject"],
    how="left")

In [31]:
df['age'] = round(df['age'])

In [32]:
df = df.drop(columns=race_cols)

### Set "Ground Truth" Attention label

In [3]:
attention_cols = [
    col for col in df.columns
    if 'attentionfilled' in col and 'self' not in col]

# use averaging to serve as a ground truth label
df['attention'] = df[attention_cols].mean(axis=1)

### Unique identifier for each subject

In [3]:
df['subject_id'] = df['group'] + "_" + df['subject'] 

### Export to Parquet

In [4]:
df.to_parquet("Data/dipser_transformed_data.parquet", engine="pyarrow", index=False)